In [1]:
from aiida.engine import run_get_node
from topworkchain import PrototypeTopWorkChain
from aiida.orm import FolderData
from aiida import load_profile
load_profile()


Profile<uuid='52dacf545d5c4d089ba5e0dd6267dcf0' name='presto'>

In [2]:
#Run the model
results, node = run_get_node(PrototypeTopWorkChain)
#Load the FolderData node containing the compressed cube files (a folder of files stored on the AiiDA database.)
cube_folder = results["cube_folder"]

03/19/2026 11:40:07 AM <144591> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
03/19/2026 11:40:07 AM <144591> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
03/19/2026 11:40:07 AM <144591> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.


uuid: c27f84bf-bc27-4f48-a618-255944ccf95c (pk: 1084)


In [3]:
import nglview as nv
import ipywidgets as widgets
import traitlets as tl
import ase.io
import io
import os
from ase.build import molecule
import tempfile

In [4]:
#Reading the (currently 1) files from the AiiDA node.
#Attempt without a temporary file (doesnt work)
# with cube_folder.open("compressed_cube", mode="rb") as cube_file:
#     raw_bytes = cube_file.read()
#     text_stream = io.StringIO(raw_bytes.decode("utf-8"))
#     byte_stream = io.BytesIO(cube_file.read())
#     data, acrolein = ase.io.cube.read_cube_data(text_stream)
#     test = nv.show_ase(acrolein)
#     orbital_component = test.add_component(byte_stream, ext="cube")

#Attempt with a temporary file.
with cube_folder.open("compressed_cube", mode="rb") as file_in:
    with open("temp_cube.cube", "wb") as file_out:
        file_out.write(file_in.read())

CUBE_PATH = "temp_cube.cube"
acrolein = ase.io.read("temp_cube.cube")

# if os.path.exists("temp_cube.cube"):
#         os.remove("temp_cube.cube")

# CUBE_PATH = "cubes/reduced_acrolein.s1.mo14a_attempt_3.cube"
# acrolein = ase.io.read("cubes/reduced_acrolein.s1.mo14a_attempt_3.cube")
test = nv.show_ase(acrolein)

# test = nv.NGLWidget()
orbital_component = test.add_component(CUBE_PATH, ext="cube")

positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
    
)
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
)
positive_color = widgets.ColorPicker(value="#3366ff", description="+ color")
negative_color = widgets.ColorPicker(value="#ff4d4d", description="- color")
show_positive = widgets.Checkbox(value=True, description="Show positive")
show_negative = widgets.Checkbox(value=True, description="Show negative")
status = widgets.HTML()

def redraw_surfaces(_=None):
    
    orbital_component.clear()

    if show_positive.value:
        orbital_component.add_surface(
            color=positive_color.value,
            isolevelType="value",
            isolevel=float(positive_level.value),
            opacity=0.5,
        )

    if show_negative.value:
        orbital_component.add_surface(
            color=negative_color.value,
            isolevelType="value",
            isolevel=-float(negative_level.value),
            opacity=0.5,
        )

    status.value = (
        f"Positive: {positive_level.value:.2e} | "
        f"Negative: {-negative_level.value:.2e}"
    )


for control in [
    positive_level,
    negative_level,
    positive_color,
    negative_color,
    show_positive,
    show_negative,
]:
    control.observe(redraw_surfaces, names="value")

redraw_surfaces()

controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        widgets.HBox([positive_level, positive_color]),
        widgets.HBox([negative_level, negative_color]),
        widgets.HBox([show_positive, show_negative]),
        status,
    ]
)

widgets.VBox([controls, test])
